# 01 Check the converted published model

Converts the published DeepVoid model
([doi:10.5281/zenodo.24261832](https://doi.org/10.5281/zenodo.24261832))
from Keras to PyTorch and checks that the PyTorch version predicts the same
thing, on the TNG300-3 subhalo grids
([doi:10.5281/zenodo.24233069](https://doi.org/10.5281/zenodo.24233069)) at
all four trained spacings:

1. class scores compared voxel by voxel with the Keras model on the same input;
2. four-class and void MCC on the test slab compared with the published
   `test_scores.json` (four-class 0.885 / 0.842 / 0.781 / 0.683 at
   $\lambda$ = 3 / 5 / 7 / 10 $h^{-1}$ Mpc).

The Keras model was trained with mixed precision and is saved that way, so
Keras computes in float16 (with float32 weights), and the published scores
were made like that. The PyTorch model is run twice: in float32 (a clean
reference) and in float16 with autocast (comparable to Keras). Small score
differences between float16 and float32 are rounding, not a conversion error.

Downloads: the model (113 MB) and five grids (four subhalo grids and the
T-web labels). Use a GPU runtime (L4 is enough) with High-RAM.

In [ ]:
# Get the code. Colab already has PyTorch, TensorFlow and Keras.
!pip -q install "deepvoid-torch[convert] @ git+https://github.com/skumagai3/deepvoid-torch"

## 1. Download the model and the data

TensorFlow by default reserves almost all of the GPU's memory as soon as it
starts, which would leave none for PyTorch in the same session. Memory
growth makes it take only what it uses; it must be set before TensorFlow
starts, so it comes first.

In [ ]:
import os
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

import json
import time

import numpy as np
import torch
import yaml
from deepvoid import data, pretrained
from deepvoid.evaluate import confusion_matrix, scores as score_cm
from deepvoid.predict import predict_volume
from deepvoid.split import slab_split

SPACINGS = [3, 5, 7, 10]
NAMES = {s: f'tng300-3/subhalos/L{s}' for s in SPACINGS}

keras_model = pretrained.load(folder='/content/deepvoid_models')
catalog = data.fetch(list(NAMES.values()) + ['tng300-3/mask'], '/content/deepvoid_data')
published = json.loads((keras_model.folder / 'test_scores.json').read_text())
print('trained spacings:', keras_model.spacings, '| cell size:', keras_model.info['cell_size'])
print('Keras layer dtype policy:', keras_model.model.get_layer('enc0_conv1').dtype_policy.name)

## 2. Convert

`convert` reads `model.keras` with Keras, copies every weight into the
PyTorch `UNet3D` (failing if any weight is left over or missing) and writes
`model.pt`.

In [ ]:
from deepvoid_torch.convert import convert

pt_path = keras_model.folder / 'model.pt'
torch_model = convert(keras_model.folder / 'model.keras', pt_path).cuda().eval()
print(torch.load(pt_path)['architecture'])

## 3. A prediction function for PyTorch

`deepvoid.predict.predict_volume` cuts the box into overlapping crops and
calls a function on each batch of them. It works in the Keras layout,
(batch, x, y, z, channels), on NumPy arrays, and expects softmax scores
back. So the PyTorch function:

- turns the batch into a tensor on the GPU and moves channels to axis 1;
- runs the model with gradients off (`torch.no_grad`: nothing is trained here);
- applies the softmax in float32 (the model returns logits) and moves the
  channels back to the last axis.

On recent NVIDIA GPUs (A100, L4) PyTorch by default does float32
convolutions in TF32, which keeps only 10 bits of the mantissa. That is
switched off here so the float32 run is a true float32 reference.

In [ ]:
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False


def torch_predict(batch, half=False):
    x = torch.from_numpy(np.ascontiguousarray(batch)).cuda().permute(0, 4, 1, 2, 3)
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16, enabled=half):
        logits = torch_model(x)
    return torch.softmax(logits.float(), dim=1).permute(0, 2, 3, 4, 1).cpu().numpy()

## 4. Predict the whole box at each spacing and compare

For each spacing: the Keras scores, then the PyTorch scores in float32 and
in float16, all on the same input. Scores are compared over the whole box;
MCC and class agreement on the test slab only, the part of the box never
used for training or for choosing the model.

In [ ]:
truth = catalog.load('tng300-3/mask').data
cfg = yaml.safe_load(open(keras_model.folder / 'config.yaml'))
test = slab_split(truth.shape[0], **cfg['data']['split'])['test'].mask(truth.shape[0])
ev = keras_model.info['evaluate']
crop = dict(crop=ev['crop'], margin=ev['margin'], batch_size=ev['batch_size'])


def test_scores(scores):
    s = score_cm(confusion_matrix(truth[test], scores[test].argmax(-1)))
    return {'mcc': s['mcc'], 'void_mcc': s['void_mcc']}


results = {}
for s in SPACINGS:
    tracers = catalog.load(NAMES[s])
    x = keras_model.inputs(tracers.data, tracers.box_size, float(s))
    t0 = time.time()
    k = predict_volume(keras_model.model.predict_on_batch, x, **crop)
    t_keras = time.time() - t0
    row = {'published': {key: published[NAMES[s]][key] for key in ('mcc', 'void_mcc')},
           'keras': test_scores(k)}
    for label, half in (('torch_fp32', False), ('torch_fp16', True)):
        t0 = time.time()
        p = predict_volume(lambda b: torch_predict(b, half), x, **crop)
        row[label] = {**test_scores(p),
                      'max_abs_diff': float(np.abs(p - k).max()),
                      'mean_abs_diff': float(np.abs(p - k).mean()),
                      'same_class_test': float((p[test].argmax(-1) == k[test].argmax(-1)).mean()),
                      'seconds': time.time() - t0}
        del p
    row['keras']['seconds'] = t_keras
    results[s] = row
    del k, x
    print(f'lambda = {s}:', json.dumps(row, indent=1))

## 5. Summary

What to expect if the conversion is right:

- Keras reproduces the published MCCs exactly (same model, same input).
- PyTorch float16 agrees with Keras on almost every test-slab voxel, and its
  MCCs match to about the third decimal; the remaining differences are
  float16 rounding, which is not done identically by the two frameworks.
- PyTorch float32 differs from Keras by about the size of float16 rounding
  (around 1e-3 in the scores), since Keras computed in float16.

A conversion mistake would instead show up as large score differences
(tenths) and MCCs far from the published ones.

In [ ]:
print(f"{'':>3} {'published':>10} {'keras':>8} {'fp32':>8} {'fp16':>8}   "
      f"{'max|diff| fp32':>14} {'fp16':>8}   {'same class fp32':>15} {'fp16':>8}")
for s, r in results.items():
    print(f"{s:>3} {r['published']['mcc']:>10.4f} {r['keras']['mcc']:>8.4f} "
          f"{r['torch_fp32']['mcc']:>8.4f} {r['torch_fp16']['mcc']:>8.4f}   "
          f"{r['torch_fp32']['max_abs_diff']:>14.2e} {r['torch_fp16']['max_abs_diff']:>8.2e}   "
          f"{r['torch_fp32']['same_class_test']:>15.6f} {r['torch_fp16']['same_class_test']:>8.6f}")
print('four-class MCC on the test slab; differences against the Keras scores')

(keras_model.folder / 'parity.json').write_text(json.dumps(results, indent=2));

## 6. Keep the converted model

Copies `model.pt` and the comparison to Drive, next to the other DeepVoid
files.

In [ ]:
import shutil
from google.colab import drive

drive.mount('/content/drive')
out = '/content/drive/MyDrive/deepvoid_model_release_torch'
os.makedirs(out, exist_ok=True)
for name in ('model.pt', 'parity.json'):
    shutil.copy2(keras_model.folder / name, out)
print('saved to', out)